# Vision Assignment P03
## Classical Feature Extraction

Notebook ini menjalankan SIFT, ORB, feature matching, homography, dan image stitching sederhana di Google Colab.

## Deskripsi library dan metode

| Komponen | Deskripsi |
|---|---|
| OpenCV | Library computer vision untuk membaca gambar, feature extraction, matching, dan stitching. |
| NumPy | Operasi array, koordinat keypoint, dan matriks homography. |
| Matplotlib | Menampilkan gambar, keypoint, hasil matching, dan panorama. |
| SIFT | Feature extractor dengan invariansi skala dan rotasi; descriptor float dicocokkan dengan L2. |
| ORB | Feature extractor cepat berbasis FAST dan BRIEF; descriptor biner dicocokkan dengan Hamming. |
| BFMatcher | Brute-force matcher untuk mencari descriptor terdekat. |
| Ratio test | Filter untuk membuang pasangan descriptor yang ambigu. |
| Homography | Matriks transformasi perspektif antara dua bidang gambar. |
| RANSAC | Estimasi robust yang membuang outlier dari pasangan keypoint. |

In [ ]:
%pip install -q opencv-contrib-python matplotlib numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

print('OpenCV:', cv2.__version__)
uploaded = files.upload()
image_name = next(iter(uploaded))
image1 = cv2.imread(image_name)
if image1 is None:
    raise ValueError('Gambar tidak dapat dibaca')
gray1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
print('Image shape:', image1.shape)

## 1. Membuat gambar kedua

Gambar kedua dibuat melalui transformasi perspektif agar matching dan stitching dapat direproduksi tanpa membutuhkan pasangan gambar eksternal. Anda juga dapat mengganti bagian ini dengan upload gambar kedua.

In [ ]:
h, w = gray1.shape
source_points = np.float32([[0, 0], [w-1, 0], [w-1, h-1], [0, h-1]])
target_points = np.float32([[.04*w, .05*h], [.95*w, 0], [w, .95*h], [0, h]])
H_transform = cv2.getPerspectiveTransform(source_points, target_points)
image2 = cv2.warpPerspective(image1, H_transform, (w, h))
gray2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)); axes[0].set_title('Gambar 1')
axes[1].imshow(cv2.cvtColor(image2, cv2.COLOR_BGR2RGB)); axes[1].set_title('Gambar 2')
for ax in axes: ax.axis('off')
plt.show()

## 2. SIFT

SIFT mendeteksi keypoint pada beberapa skala dan menghasilkan descriptor float. Jarak L2 digunakan saat matching.

In [ ]:
sift = cv2.SIFT_create(nfeatures=2000)
kp1_sift, des1_sift = sift.detectAndCompute(gray1, None)
kp2_sift, des2_sift = sift.detectAndCompute(gray2, None)
print('SIFT keypoints:', len(kp1_sift), len(kp2_sift))
print('SIFT descriptor:', None if des1_sift is None else des1_sift.shape)

sift_visual = cv2.drawKeypoints(image1, kp1_sift, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
plt.figure(figsize=(12, 6)); plt.imshow(cv2.cvtColor(sift_visual, cv2.COLOR_BGR2RGB)); plt.title('SIFT keypoints'); plt.axis('off'); plt.show()

## 3. ORB

ORB menggabungkan FAST dan BRIEF. Descriptor biner dicocokkan dengan jarak Hamming dan biasanya lebih cepat daripada SIFT.

In [ ]:
orb = cv2.ORB_create(nfeatures=2000)
kp1_orb, des1_orb = orb.detectAndCompute(gray1, None)
kp2_orb, des2_orb = orb.detectAndCompute(gray2, None)
print('ORB keypoints:', len(kp1_orb), len(kp2_orb))
print('ORB descriptor:', None if des1_orb is None else des1_orb.shape)

## 4. Feature matching dan ratio test

In [ ]:
def ratio_matches(des1, des2, norm, ratio=.75):
    if des1 is None or des2 is None:
        return []
    matcher = cv2.BFMatcher(norm)
    pairs = matcher.knnMatch(des1, des2, k=2)
    return [a for a, b in pairs if a.distance < ratio * b.distance]

sift_good = ratio_matches(des1_sift, des2_sift, cv2.NORM_L2)
orb_good = ratio_matches(des1_orb, des2_orb, cv2.NORM_HAMMING)
print('SIFT good matches:', len(sift_good))
print('ORB good matches:', len(orb_good))

def display_matches(kp1, kp2, matches, title):
    output = cv2.drawMatches(image1, kp1, image2, kp2, matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(16, 7)); plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB)); plt.title(title); plt.axis('off'); plt.show()

display_matches(kp1_sift, kp2_sift, sift_good, 'SIFT matching')
display_matches(kp1_orb, kp2_orb, orb_good, 'ORB matching')

## 5. Homography dan image stitching

In [ ]:
if len(sift_good) < 4:
    raise ValueError('SIFT match kurang dari empat; homography tidak dapat dihitung.')

src_pts = np.float32([kp1_sift[m.queryIdx].pt for m in sift_good]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2_sift[m.trainIdx].pt for m in sift_good]).reshape(-1, 1, 2)
H, mask = cv2.findHomography(dst_pts, src_pts, cv2.RANSAC, 5.0)
if H is None:
    raise ValueError('Homography gagal dihitung.')

corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
warped = cv2.perspectiveTransform(corners, H)
all_corners = np.concatenate([corners, warped], axis=0)
xmin, ymin = np.floor(all_corners.min(axis=0).ravel()).astype(int)
xmax, ymax = np.ceil(all_corners.max(axis=0).ravel()).astype(int)
T = np.array([[1, 0, -xmin], [0, 1, -ymin], [0, 0, 1]], dtype=float)
panorama = cv2.warpPerspective(image2, T @ H, (xmax-xmin, ymax-ymin))
panorama[-ymin:h-ymin, -xmin:w-xmin] = image1
print('Homography inliers:', int(mask.sum()), '/', len(mask))
plt.figure(figsize=(16, 8)); plt.imshow(cv2.cvtColor(panorama, cv2.COLOR_BGR2RGB)); plt.title('Image stitching'); plt.axis('off'); plt.show()

In [ ]:
# Validasi eksperimen
assert len(kp1_sift) > 0 and len(kp1_orb) > 0
assert len(sift_good) >= 4
assert H.shape == (3, 3)
print('Validasi feature extraction dan stitching: PASS')

## 6. Analisis dan kesimpulan otomatis

In [ ]:
sift_inlier_ratio = int(mask.sum()) / len(sift_good) if len(sift_good) else 0
analysis = f'''
ANALISIS OTOMATIS
SIFT mendeteksi {len(kp1_sift)} dan {len(kp2_sift)} keypoint pada dua gambar,
sedangkan ORB mendeteksi {len(kp1_orb)} dan {len(kp2_orb)} keypoint.
Setelah ratio test, SIFT menghasilkan {len(sift_good)} good matches dan ORB
menghasilkan {len(orb_good)} good matches. Homography memiliki {int(mask.sum())}
inlier dari {len(mask)} match SIFT, sehingga inlier ratio sebesar {sift_inlier_ratio:.3f}.
SIFT cenderung lebih robust, sedangkan ORB lebih cepat dan hemat memori.
'''
conclusion = f'''
KESIMPULAN OTOMATIS
Eksperimen berhasil mendeteksi fitur menggunakan SIFT dan ORB, mencocokkan
descriptor dengan distance metric yang sesuai, lalu menghitung homography
menggunakan RANSAC. Dengan {int(mask.sum())} inlier, image stitching dapat
dibentuk dari dua gambar. SIFT sesuai untuk akurasi dan invariansi, sedangkan
ORB sesuai untuk aplikasi real-time. Hasil tetap bergantung pada tekstur,
overlap, kualitas gambar, dan jumlah keypoint.
'''
print(analysis)
print(conclusion)